# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.** The decision is "which content gets reviewed first," not "is this one page declining or not" — reviewer capacity is fixed, so what matters is the order of the queue, not any single row's verdict. Mechanically, the reference pipeline fits a classifier (`random_forest` on `is_declining_label`), but its output is only ever consumed as a continuous priority score (`predict_proba`) that gets sorted — the classifier is the mechanism, ranking is the task. That distinction has teeth: framing this as classification would pull toward ROC-AUC/accuracy, metrics that score the whole distribution, when the only thing that matters for a 50-slot reviewer is precision@50 — whether the top of the queue is actually worth their time.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy, not observed.** `is_declining_label = 1` when `trend_direction == "down"`, and `trend_direction` is a threshold on `trend_pct = (impressions_last_30d − impressions_prev_30d) / impressions_prev_30d`, with "down" meaning anything below −20%. Two things make this a defined rule rather than an observed outcome: (1) it's an arbitrary cutoff — a page at −19% and one at −21% are nearly identical but land on opposite sides of the label; (2) `last_30d` and `prev_30d` are both already inside the static snapshot you have today — nothing about this label required waiting to see what happens next. A genuinely observed target would use the warehouse's `report_date`-spanning fact table: did this `content_id`'s traffic actually keep declining (or recover) in the 30–60 days after a chosen decision date — a future window you'd have to wait for, not one already sitting in the CSV.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: precision@50.** K = 50 because that matches the reviewer's realistic weekly review capacity (per w01, a team can look at ~20–50 pages a cycle) — not an arbitrary round number, it's the actual bottleneck the decision is built around. Precision@50 = 0.74 means: of the top 50 pages the queue recommends, 37 of them (74%) are genuine review-worthy candidates, and ~13 (26%) would be wasted reviewer time. "Good" for this project means beating the fixed-rule baseline of 0.24 (12/50) — the model's 0.74 already clears that bar by 3x, and any future work should be judged against 0.74 as the number to beat, not some abstract "high accuracy."

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# is_declining_label doesn't exist in the raw CSV -- it's added by the prep step.
# Build it the same way the pipeline does: trend_direction == "down".
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

unit_of_analysis = df[
    ["content_id", "client_id", "impressions_90d", "avg_position", "ctr", "trend_direction", "is_declining_label"]
]
print(f"{len(unit_of_analysis):,} rows -- one row = one content item (content_id) in its trailing-90-day snapshot")
unit_of_analysis.head()

30,000 rows -- one row = one content item (content_id) in its trailing-90-day snapshot


,content_id,client_id,impressions_90d,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A fixed rule can't scale across heterogeneity or interactions.** This dataset spans 32 different clients — a hand-picked threshold like `ctr < 0.5` implicitly assumes 0.5 means the same thing everywhere, but different clients have different traffic scales and baseline CTRs, so one global cutoff will misfire for some of them. Worse, the signals interact: a low CTR at `avg_position = 3` (a page ranking well but still under-clicked — a real opportunity) means something very different from a low CTR at `avg_position = 40` (barely visible, so low clicks are expected, not a signal). A chain of independent if-statements checks each signal in isolation and can't represent that conditional relationship. This is exactly the gap the numbers confirm: the fixed weighted rule hit precision@50 = 0.24, while a random forest — trained to learn those non-linear, client-varying interactions automatically — hit 0.74 on the same signals. The pattern is real; it's just too tangled for hand-written thresholds to capture.

## Self-check

Before you submit, confirm each line honestly:

- [* ] Every section above is filled — markdown thinking AND the code that backs it
- [* ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [* ] No client names, URLs, or private queries anywhere
- [* ] My claims use careful words: observed, measured, directional, decision-support
- [* ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.